### 환경 설정

In [2]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from itertools import product
from sklearn.metrics import roc_auc_score

train = pd.read_csv('../data/raw/train.csv')
y = train['임신 성공 여부']

### 튜닝 앙상블 — Best OOF: 0.7410 (리더보드: 0.7410583256)

In [ ]:
lgbm_oof = np.load('../saved_models/exp08_tuned_lgbm_oof.npy')
xgb_oof  = np.load('../saved_models/exp09_tuned_xgb_oof.npy')
cat_oof  = np.load('../saved_models/exp10_tuned_cat_oof.npy')

lgbm_test = np.load('../saved_models/exp08_tuned_lgbm_test.npy')
xgb_test  = np.load('../saved_models/exp09_tuned_xgb_test.npy')
cat_test  = np.load('../saved_models/exp10_tuned_cat_test.npy')

print(f'LightGBM OOF : {roc_auc_score(y, lgbm_oof):.4f}')
print(f'XGBoost  OOF : {roc_auc_score(y, xgb_oof):.4f}')
print(f'CatBoost OOF : {roc_auc_score(y, cat_oof):.4f}')

best_score   = 0
best_weights = None

for w1, w2, w3 in product(range(0, 11), repeat=3):
    if w1 + w2 + w3 == 0:
        continue
    w1, w2, w3 = w1/10, w2/10, w3/10
    if abs(w1 + w2 + w3 - 1.0) > 1e-6:
        continue
    oof   = w1*lgbm_oof + w2*xgb_oof + w3*cat_oof
    score = roc_auc_score(y, oof)
    if score > best_score:
        best_score   = score
        best_weights = (w1, w2, w3)

print(f'Best Weights  — LightGBM: {best_weights[0]}, XGBoost: {best_weights[1]}, CatBoost: {best_weights[2]}')
print(f'Best OOF ROC-AUC : {best_score:.4f}')

### MICE 앙상블 — Best OOF: 0.7403

In [ ]:
lgbm_oof = np.load('../saved_models/exp10_mice_lgbm_oof.npy')
xgb_oof  = np.load('../saved_models/exp11_mice_xgb_oof.npy')
cat_oof  = np.load('../saved_models/exp12_mice_cat_oof.npy')
mlp_oof  = np.load('../saved_models/exp13_mice_mlp_oof.npy')

lgbm_test = np.load('../saved_models/exp10_mice_lgbm_test.npy')
xgb_test  = np.load('../saved_models/exp11_mice_xgb_test.npy')
cat_test  = np.load('../saved_models/exp12_mice_cat_test.npy')
mlp_test  = np.load('../saved_models/exp13_mice_mlp_test.npy')

print(f'LightGBM OOF : {roc_auc_score(y, lgbm_oof):.4f}')
print(f'XGBoost  OOF : {roc_auc_score(y, xgb_oof):.4f}')
print(f'CatBoost OOF : {roc_auc_score(y, cat_oof):.4f}')
print(f'MLP      OOF : {roc_auc_score(y, mlp_oof):.4f}')

best_score   = 0
best_weights = None

for w1, w2, w3, w4 in product(range(0, 11), repeat=4):
    if w1 + w2 + w3 + w4 == 0:
        continue
    total = w1 + w2 + w3 + w4
    oof   = (w1*lgbm_oof + w2*xgb_oof + w3*cat_oof + w4*mlp_oof) / total
    score = roc_auc_score(y, oof)
    if score > best_score:
        best_score   = score
        best_weights = (w1/total, w2/total, w3/total, w4/total)

print(f'Best Weights — LightGBM: {best_weights[0]:.2f}, XGBoost: {best_weights[1]:.2f}, CatBoost: {best_weights[2]:.2f}, MLP: {best_weights[3]:.2f}')
print(f'Best OOF ROC-AUC : {best_score:.4f}')

### MICE + balanced 앙상블

In [ ]:
lgbm_oof = np.load('../saved_models/exp14_mice_lgbm_balanced_oof.npy')
xgb_oof  = np.load('../saved_models/exp15_mice_xgb_balanced_oof.npy')
cat_oof  = np.load('../saved_models/exp16_mice_cat_balanced_oof.npy')

lgbm_test = np.load('../saved_models/exp14_mice_lgbm_balanced_test.npy')
xgb_test  = np.load('../saved_models/exp15_mice_xgb_balanced_test.npy')
cat_test  = np.load('../saved_models/exp16_mice_cat_balanced_test.npy')

print(f'LightGBM OOF : {roc_auc_score(y, lgbm_oof):.4f}')
print(f'XGBoost  OOF : {roc_auc_score(y, xgb_oof):.4f}')
print(f'CatBoost OOF : {roc_auc_score(y, cat_oof):.4f}')

best_score   = 0
best_weights = None

for w1, w2, w3 in product(range(0, 11), repeat=3):
    if w1 + w2 + w3 == 0:
        continue
    w1, w2, w3 = w1/10, w2/10, w3/10
    if abs(w1 + w2 + w3 - 1.0) > 1e-6:
        continue
    oof   = w1*lgbm_oof + w2*xgb_oof + w3*cat_oof
    score = roc_auc_score(y, oof)
    if score > best_score:
        best_score   = score
        best_weights = (w1, w2, w3)

print(f'Best Weights — LightGBM: {best_weights[0]}, XGBoost: {best_weights[1]}, CatBoost: {best_weights[2]}')
print(f'Best OOF ROC-AUC : {best_score:.4f}')

In [2]:
import numpy as np
import json
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score
import pandas as pd

# OOF 로드
lgbm_oofs = [np.load(f'../saved_models/exp_tuned_lgbm_seed{seed}_oof.npy') for seed in [42, 123, 456]]
xgb_oofs  = [np.load(f'../saved_models/exp_tuned_xgb_seed{seed}_oof.npy')  for seed in [42, 123, 456]]
cat_oofs  = [np.load(f'../saved_models/exp_tuned_cat_seed{seed}_oof.npy')  for seed in [42, 123, 456]]

lgbm_oof = np.mean(lgbm_oofs, axis=0)
xgb_oof  = np.mean(xgb_oofs,  axis=0)
cat_oof  = np.mean(cat_oofs,  axis=0)

lgbm_tests = [np.load(f'../saved_models/exp_tuned_lgbm_seed{seed}_test.npy') for seed in [42, 123, 456]]
xgb_tests  = [np.load(f'../saved_models/exp_tuned_xgb_seed{seed}_test.npy')  for seed in [42, 123, 456]]
cat_tests  = [np.load(f'../saved_models/exp_tuned_cat_seed{seed}_test.npy')  for seed in [42, 123, 456]]

lgbm_test = np.mean(lgbm_tests, axis=0)
xgb_test  = np.mean(xgb_tests,  axis=0)
cat_test  = np.mean(cat_tests,  axis=0)

train = pd.read_csv('../data/raw/train.csv')
y = train['임신 성공 여부'].values

def neg_roc(weights):
    w = np.array(weights)
    w = w / w.sum()
    pred = w[0]*lgbm_oof + w[1]*xgb_oof + w[2]*cat_oof
    return -roc_auc_score(y, pred)

result = minimize(neg_roc, [1/3, 1/3, 1/3], method='Nelder-Mead')
best_w = result.x / result.x.sum()
print(f'Best weights - LGBM: {best_w[0]:.3f}, XGB: {best_w[1]:.3f}, CAT: {best_w[2]:.3f}')
print(f'Best OOF ROC-AUC: {-result.fun:.4f}')

test_pred = best_w[0]*lgbm_test + best_w[1]*xgb_test + best_w[2]*cat_test

submission = pd.read_csv('../data/raw/sample_submission.csv')
submission['probability'] = test_pred
submission.to_csv('../submissions/submission_ensemble_tuned.csv', index=False)
print('저장 완료')

Best weights - LGBM: 0.085, XGB: 0.590, CAT: 0.325
Best OOF ROC-AUC: 0.7405
저장 완료


In [3]:
# 시드 앙상블 OOF 로드
lgbm_oofs = [np.load(f'../saved_models/exp_tuned_lgbm_seed{seed}_oof.npy') for seed in [42, 123, 456]]
xgb_oofs  = [np.load(f'../saved_models/exp_tuned_xgb_seed{seed}_oof.npy')  for seed in [42, 123, 456]]
cat_oofs  = [np.load(f'../saved_models/exp_tuned_cat_seed{seed}_oof.npy')  for seed in [42, 123, 456]]
mlp_oofs  = [np.load(f'../saved_models/exp{17+i}_mlp_seed{seed}_oof.npy')  for i, seed in enumerate([42, 123, 456])]

lgbm_oof = np.mean(lgbm_oofs, axis=0)
xgb_oof  = np.mean(xgb_oofs,  axis=0)
cat_oof  = np.mean(cat_oofs,  axis=0)
mlp_oof  = np.mean(mlp_oofs,  axis=0)

lgbm_tests = [np.load(f'../saved_models/exp_tuned_lgbm_seed{seed}_test.npy') for seed in [42, 123, 456]]
xgb_tests  = [np.load(f'../saved_models/exp_tuned_xgb_seed{seed}_test.npy')  for seed in [42, 123, 456]]
cat_tests  = [np.load(f'../saved_models/exp_tuned_cat_seed{seed}_test.npy')  for seed in [42, 123, 456]]
mlp_tests  = [np.load(f'../saved_models/exp{17+i}_mlp_seed{seed}_test.npy')  for i, seed in enumerate([42, 123, 456])]

lgbm_test = np.mean(lgbm_tests, axis=0)
xgb_test  = np.mean(xgb_tests,  axis=0)
cat_test  = np.mean(cat_tests,  axis=0)
mlp_test  = np.mean(mlp_tests,  axis=0)

print(f'LightGBM OOF : {roc_auc_score(y, lgbm_oof):.4f}')
print(f'XGBoost  OOF : {roc_auc_score(y, xgb_oof):.4f}')
print(f'CatBoost OOF : {roc_auc_score(y, cat_oof):.4f}')
print(f'MLP      OOF : {roc_auc_score(y, mlp_oof):.4f}')

best_score   = 0
best_weights = None

for w1, w2, w3, w4 in product(range(0, 11), repeat=4):
    if w1 + w2 + w3 + w4 == 0:
        continue
    total = w1 + w2 + w3 + w4
    oof   = (w1*lgbm_oof + w2*xgb_oof + w3*cat_oof + w4*mlp_oof) / total
    score = roc_auc_score(y, oof)
    if score > best_score:
        best_score   = score
        best_weights = (w1/total, w2/total, w3/total, w4/total)

print(f'Best Weights — LGBM: {best_weights[0]:.2f}, XGB: {best_weights[1]:.2f}, CAT: {best_weights[2]:.2f}, MLP: {best_weights[3]:.2f}')
print(f'Best OOF ROC-AUC: {best_score:.4f}')

test_pred = best_weights[0]*lgbm_test + best_weights[1]*xgb_test + best_weights[2]*cat_test + best_weights[3]*mlp_test

submission = pd.read_csv('../data/raw/sample_submission.csv')
submission['probability'] = test_pred
submission.to_csv('../submissions/submission_ensemble_final.csv', index=False)
print('저장 완료')

LightGBM OOF : 0.7402
XGBoost  OOF : 0.7404
CatBoost OOF : 0.7402
MLP      OOF : 0.7393
Best Weights — LGBM: 0.00, XGB: 0.60, CAT: 0.13, MLP: 0.27
Best OOF ROC-AUC: 0.7406
저장 완료


In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb

train = pd.read_csv('../data/raw/train.csv')
y = train['임신 성공 여부'].values

# 부스팅 그룹
lgbm_oofs = [np.load(f'../saved_models/exp_tuned_lgbm_seed{seed}_oof.npy') for seed in [42, 123, 456]]
xgb_oofs  = [np.load(f'../saved_models/exp_tuned_xgb_seed{seed}_oof.npy')  for seed in [42, 123, 456]]
cat_oofs  = [np.load(f'../saved_models/exp_tuned_cat_seed{seed}_oof.npy')  for seed in [42, 123, 456]]

lgbm_tests = [np.load(f'../saved_models/exp_tuned_lgbm_seed{seed}_test.npy') for seed in [42, 123, 456]]
xgb_tests  = [np.load(f'../saved_models/exp_tuned_xgb_seed{seed}_test.npy')  for seed in [42, 123, 456]]
cat_tests  = [np.load(f'../saved_models/exp_tuned_cat_seed{seed}_test.npy')  for seed in [42, 123, 456]]

# 배깅 그룹
et_oofs  = [np.load(f'../saved_models/exp24_et_seed{seed}_oof.npy')  for seed in [42, 123, 456]]
et_tests = [np.load(f'../saved_models/exp24_et_seed{seed}_test.npy') for seed in [42, 123, 456]]

# 딥러닝 그룹
mlp_oofs    = [np.load(f'../saved_models/exp{17 + [42,123,456].index(seed)}_mlp_seed{seed}_oof.npy')    for seed in [42, 123, 456]]
tabnet_oofs = [np.load(f'../saved_models/exp{20 + [42,123,456].index(seed)}_tabnet_seed{seed}_oof.npy') for seed in [42, 123, 456]]

mlp_tests    = [np.load(f'../saved_models/exp{17 + [42,123,456].index(seed)}_mlp_seed{seed}_test.npy')    for seed in [42, 123, 456]]
tabnet_tests = [np.load(f'../saved_models/exp{20 + [42,123,456].index(seed)}_tabnet_seed{seed}_test.npy') for seed in [42, 123, 456]]

# 그룹별 평균
lgbm_oof = np.mean(lgbm_oofs, axis=0)
xgb_oof  = np.mean(xgb_oofs,  axis=0)
cat_oof  = np.mean(cat_oofs,  axis=0)
et_oof   = np.mean(et_oofs,   axis=0)
mlp_oof  = np.mean(mlp_oofs,  axis=0)
tab_oof  = np.mean(tabnet_oofs, axis=0)

lgbm_test = np.mean(lgbm_tests, axis=0)
xgb_test  = np.mean(xgb_tests,  axis=0)
cat_test  = np.mean(cat_tests,  axis=0)
et_test   = np.mean(et_tests,   axis=0)
mlp_test  = np.mean(mlp_tests,  axis=0)
tab_test  = np.mean(tabnet_tests, axis=0)

print(f'LightGBM OOF : {roc_auc_score(y, lgbm_oof):.4f}')
print(f'XGBoost  OOF : {roc_auc_score(y, xgb_oof):.4f}')
print(f'CatBoost OOF : {roc_auc_score(y, cat_oof):.4f}')
print(f'ET       OOF : {roc_auc_score(y, et_oof):.4f}')
print(f'MLP      OOF : {roc_auc_score(y, mlp_oof):.4f}')
print(f'TabNet   OOF : {roc_auc_score(y, tab_oof):.4f}')

# 메타 피처
meta_train = np.column_stack([lgbm_oof, xgb_oof, cat_oof, et_oof, mlp_oof, tab_oof])
meta_test  = np.column_stack([lgbm_test, xgb_test, cat_test, et_test, mlp_test, tab_test])

# 메타 모델 비교
meta_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'LightGBM':           lgb.LGBMClassifier(n_estimators=100, verbose=-1, random_state=42),
    'XGBoost':            xgb.XGBClassifier(n_estimators=100, verbosity=0, random_state=42),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_score = 0
best_name  = None
best_test_pred = None

for model_name, meta_model in meta_models.items():
    meta_oof_pred   = np.zeros(len(y))
    meta_test_preds = np.zeros((meta_test.shape[0], 5))

    for i, (tr_idx, val_idx) in enumerate(skf.split(meta_train, y)):
        meta_model.fit(meta_train[tr_idx], y[tr_idx])
        meta_oof_pred[val_idx] = meta_model.predict_proba(meta_train[val_idx])[:, 1]
        meta_test_preds[:, i]  = meta_model.predict_proba(meta_test)[:, 1]

    score = roc_auc_score(y, meta_oof_pred)
    print(f'{model_name} Stacking OOF ROC-AUC: {score:.4f}')

    if score > best_score:
        best_score     = score
        best_name      = model_name
        best_test_pred = meta_test_preds.mean(axis=1)

print(f'\nBest 메타 모델: {best_name} ({best_score:.4f})')

submission = pd.read_csv('../data/raw/sample_submission.csv')
submission['probability'] = best_test_pred
submission.to_csv('../submissions/submission_stacking.csv', index=False)
print('저장 완료')

LightGBM OOF : 0.7402
XGBoost  OOF : 0.7404
CatBoost OOF : 0.7402
ET       OOF : 0.7366
MLP      OOF : 0.7393
TabNet   OOF : 0.7392
LogisticRegression Stacking OOF ROC-AUC: 0.7405


c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-pack

LightGBM Stacking OOF ROC-AUC: 0.7397
XGBoost Stacking OOF ROC-AUC: 0.7364

Best 메타 모델: LogisticRegression (0.7405)
저장 완료


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

train = pd.read_csv('../data/raw/train.csv')
y = train['임신 성공 여부'].values

lgbm_oof  = np.load('../saved_models/exp_tuned_lgbm_seed42_oof.npy')
xgb_oof   = np.load('../saved_models/exp_tuned_xgb_seed42_oof.npy')
cat_oof   = np.load('../saved_models/exp_tuned_cat_seed42_oof.npy')
ftt_oof   = np.load('../saved_models/exp25_fttransformer_seed42_oof.npy')
saint_oof = np.load('../saved_models/exp26_saint_seed42_oof.npy')

lgbm_test  = np.load('../saved_models/exp_tuned_lgbm_seed42_test.npy')
xgb_test   = np.load('../saved_models/exp_tuned_xgb_seed42_test.npy')
cat_test   = np.load('../saved_models/exp_tuned_cat_seed42_test.npy')
ftt_test   = np.load('../saved_models/exp25_fttransformer_seed42_test.npy')
saint_test = np.load('../saved_models/exp26_saint_seed42_test.npy')

print(f'LightGBM OOF : {roc_auc_score(y, lgbm_oof):.4f}')
print(f'XGBoost  OOF : {roc_auc_score(y, xgb_oof):.4f}')
print(f'CatBoost OOF : {roc_auc_score(y, cat_oof):.4f}')
print(f'FTT      OOF : {roc_auc_score(y, ftt_oof):.4f}')
print(f'SAINT    OOF : {roc_auc_score(y, saint_oof):.4f}')

# ── 방법 1: Nelder-Mead 가중 앙상블 ──
def neg_roc(weights):
    w = np.array(weights)
    w = w / w.sum()
    pred = w[0]*lgbm_oof + w[1]*xgb_oof + w[2]*cat_oof + w[3]*ftt_oof + w[4]*saint_oof
    return -roc_auc_score(y, pred)

result   = minimize(neg_roc, [0.2, 0.2, 0.2, 0.2, 0.2], method='Nelder-Mead')
best_w   = result.x / result.x.sum()
print(f'\n[방법 1] Nelder-Mead')
print(f'Best Weights — LGBM: {best_w[0]:.3f}, XGB: {best_w[1]:.3f}, CAT: {best_w[2]:.3f}, FTT: {best_w[3]:.3f}, SAINT: {best_w[4]:.3f}')
print(f'Best OOF ROC-AUC : {-result.fun:.4f}')

test_pred_grid = best_w[0]*lgbm_test + best_w[1]*xgb_test + best_w[2]*cat_test + best_w[3]*ftt_test + best_w[4]*saint_test

submission = pd.read_csv('../data/raw/sample_submission.csv')
submission['probability'] = test_pred_grid
submission.to_csv('../submissions/submission_grid_ftt_saint.csv', index=False)
print('저장 완료: submission_grid_ftt_saint.csv')

# ── 방법 2: 로지스틱 회귀 스태킹 ──
meta_train = np.column_stack([lgbm_oof, xgb_oof, cat_oof, ftt_oof, saint_oof])
meta_test  = np.column_stack([lgbm_test, xgb_test, cat_test, ftt_test, saint_test])

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
meta_oof_pred   = np.zeros(len(y))
meta_test_preds = np.zeros((meta_test.shape[0], 10))

for i, (tr_idx, val_idx) in enumerate(skf.split(meta_train, y)):
    meta_model = LogisticRegression(max_iter=1000)
    meta_model.fit(meta_train[tr_idx], y[tr_idx])
    meta_oof_pred[val_idx] = meta_model.predict_proba(meta_train[val_idx])[:, 1]
    meta_test_preds[:, i]  = meta_model.predict_proba(meta_test)[:, 1]

print(f'\n[방법 2] 로지스틱 스태킹')
print(f'Stacking OOF ROC-AUC: {roc_auc_score(y, meta_oof_pred):.4f}')

test_pred_stack = meta_test_preds.mean(axis=1)

submission['probability'] = test_pred_stack
submission.to_csv('../submissions/submission_stacking_ftt_saint.csv', index=False)
print('저장 완료: submission_stacking_ftt_saint.csv')

LightGBM OOF : 0.7399
XGBoost  OOF : 0.7401
CatBoost OOF : 0.7399
FTT      OOF : 0.7393
SAINT    OOF : 0.7386


KeyboardInterrupt: 